# Train HeadScan-Lite on Headspace / LYHM in Google Colab

Use a GPU runtime and choose **Runtime → Run all**. This notebook now uses **Headspace / LYHM as the primary real training dataset**.

## What you need from Headspace

Request/download the licensed **FLAME registrations** package and the **3dMD raw PNG + TKA camera package**. Headspace access is for university-based non-commercial research/education and must be requested under the dataset agreement.

Put the downloaded archives (or extracted folders) anywhere under:

`MyDrive/head_model/headspace/`

You do **not** need to manually rename or convert them. The Colab script scans that folder, extracts only registered FLAME `.obj` files and `*C.png` color-camera images, pairs them by the Headspace subject ID such as `00001`, converts the registered geometry to millimetres, checks topology consistency, creates deterministic train/validation manifests, and caches the prepared result as `MyDrive/head_model/headspace_prepared_v1.zip`.

The first run may spend substantial time reading the licensed source archives. Later sessions restore the prepared cache instead, which is important for free-Colab runtimes.

Headspace geometry-only training uses the registered FLAME target mesh directly. The dense normal/depth/mask heads are skipped during this training mode, reducing VRAM and compute. Input views are treated as an unordered set; the converter intentionally does not invent yaw/pitch angles from camera filenames when calibrated angle information is not parsed.

Persistent checkpoints go to `MyDrive/head_model/headscan-lite-headspace`. `last.pt` is saved every epoch, `best.pt` when validation mean vertex error improves, and numbered checkpoints are saved every 5 epochs. Resume state includes optimizer, scheduler and AMP scaler.

Default real training: 256×256, up to 8 view slots, random available-view subsets per identity, batch size 2, up to 128 PCA components, ImageNet-pretrained MobileNetV3-Small, 3-layer/4-head variable-view Transformer, and AMP.

If no licensed Headspace folder exists, the notebook runs a tiny **synthetic smoke test** only to verify the pipeline. Those weights are not a usable head scanner.

After training, the geometry-only mobile path is exported to ONNX and returns metric mesh vertices, normalized shape coefficients and learned per-view quality weights.

In [ ]:
import pathlib, shutil, subprocess
bootstrap = pathlib.Path('/content/head_model_bootstrap')
if bootstrap.exists():
    shutil.rmtree(bootstrap)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Persie0/Head_model.git', str(bootstrap)], check=True)
subprocess.run(['git', '-C', str(bootstrap), 'rev-parse', '--short', 'HEAD'], check=True)
script = bootstrap / 'colab' / 'train_headscan_lite_colab.py'
source = script.read_text(encoding='utf-8')
required = ['HEADSCAN_COLAB_VERSION', 'v2-headspace', 'HEADSPACE_DRIVE_DIR', 'prepare_headspace', 'run_with_heartbeat']
if not all(token in source for token in required):
    raise RuntimeError('Fresh clone does not contain the expected current Headspace Colab trainer. Stop instead of running stale code.')
exec(compile(source, str(script), 'exec'), {'__name__': '__main__'})
